# B8 — Model Card : XGBoost maintenance prédictive (B11-GKF)

**Prérequis** : `TP11.ipynb` exécuté (étude Optuna + évaluation GroupKFold), base
PostgreSQL `indusense_db` accessible, `DL/TP9.ipynb` exécuté (artefact MLflow).

Ce notebook documente le **second modèle** couvert par le module B8 — celui de
maintenance prédictive (voir `DL/TP8.ipynb` pour l'auto-encodeur de détection
d'anomalies visuelles, déjà documenté séparément).

> **Mise à jour post-TP9** : B11-GKF a désormais un **artefact persisté** (MLflow,
> `DL/TP9.ipynb`) — ce n'était pas le cas à la première version de cette card. §1.5
> interroge le tracking store en direct pour rester correct dans le temps.

| § | Contenu |
|---|---|
| §1 | Rassembler les éléments — données, modèle, métriques recalculées en direct, artefact MLflow |
| §2 | Le template Hugging Face officiel |
| §3 | Usage (Direct / Downstream / Out-of-Scope), biais, risques, limites |
| §4 | Évaluation, impact environnemental, version, contact — génération finale |


## §0 — Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    confusion_matrix, precision_score, recall_score,
)
from xgboost import XGBClassifier
from codecarbon import EmissionsTracker
import mlflow

from huggingface_hub import ModelCard, ModelCardData
from huggingface_hub.repocard_data import EvalResult

RANDOM_STATE = 42
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

print(f"Template HF officiel : {ModelCard.default_template_path}")


Template HF officiel : C:\Users\Aelion\py-init\ML\.venv\Lib\site-packages\huggingface_hub\templates\modelcard_template.md


## §1 — Rassembler les éléments

### 1.1 — Données (même requête, même split que TP11)

In [2]:
url = URL.create(
    drivername="postgresql+psycopg2",
    username="indusense_user",
    password="ThEP@ssW0rd",
    host="localhost", port=5432, database="indusense_db",
)
engine = create_engine(url)
df = pd.read_sql(
    "SELECT * FROM gold_machine_hourly_feature ORDER BY machine_id, window_start",
    engine
)

TARGET = "label_failure_next_24h"
LEAKAGE_COLS = [
    "machine_id", "ingestion_batch_id", "window_start", "window_end", "split_set",
    "label_failure_next_6h", "label_failure_next_12h", "label_failure_next_48h",
    TARGET, "feature_row_id",
]
FEATURE_COLS = [c for c in df.columns if c not in LEAKAGE_COLS]

trainval_df = df[df["split_set"].isin(["train", "validation"])].copy()
test_df     = df[df["split_set"] == "test"].copy()

X_tv, y_tv   = trainval_df[FEATURE_COLS], trainval_df[TARGET]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET]
groups = trainval_df["machine_id"]

SPW_GLOBAL = round((y_tv == 0).sum() / (y_tv == 1).sum(), 2)
N_MACHINES = groups.nunique()
N_FEATURES = len(FEATURE_COLS)

print(f"Features : {N_FEATURES} | Machines : {N_MACHINES}")
print(f"Train+val : {len(X_tv):,} | Test : {len(X_test):,}")
print(f"scale_pos_weight global : {SPW_GLOBAL}")


Features : 69 | Machines : 15
Train+val : 112,996 | Test : 19,944
scale_pos_weight global : 27.12


### 1.2 — Ré-entraînement avec les hyperparamètres figés (Optuna, `TP11.ipynb`)

Les hyperparamètres ci-dessous sont ceux retenus par l'étude Optuna (30 essais, objectif
PR-AUC en GroupKFold(5)) rejouée dans `TP11.ipynb` — reproduits ici à l'identique, pas
recopiés d'un ancien run non vérifiable. On instrumente ce ré-entraînement avec
**CodeCarbon**, jamais fait auparavant côté ML (contrairement au DL, cf. `DL/TP6.ipynb`).

In [3]:
BEST_PARAMS = {
    "n_estimators":     287,
    "max_depth":        9,
    "learning_rate":    0.02887139049912187,
    "subsample":        0.8252019146348859,
    "colsample_bytree": 0.5736306798333981,
    "min_child_weight": 11,
    "reg_alpha":        0.01918528344873483,
    "reg_lambda":       0.6228756133555158,
    "scale_pos_weight": SPW_GLOBAL,
    "random_state":     RANDOM_STATE,
    "verbosity":        0,
}

tracker = EmissionsTracker(
    project_name="ml_b11_gkf_refit",
    output_dir=str(ARTIFACTS_DIR),
    measure_power_secs=1,
    log_level="error",
    save_to_file=True,
)
tracker.start()

pipe_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   XGBClassifier(**BEST_PARAMS)),
])
pipe_final.fit(X_tv, y_tv)

emissions_kg = tracker.stop()
carbon_data = tracker.final_emissions_data

print(f"Émissions   : {emissions_kg * 1000:.4f} gCO2eq")
print(f"Énergie     : {carbon_data.energy_consumed * 1000:.3f} Wh")
print(f"Durée       : {carbon_data.duration:.1f} s")
print(f"Mix détecté : {carbon_data.country_name} ({carbon_data.country_iso_code})")


[codecarbon WARNING @ 14:28:23] Multiple instances of codecarbon are allowed to run at the same time.


Émissions   : 0.0027 gCO2eq
Énergie     : 0.048 Wh
Durée       : 4.5 s
Mix détecté : France (FRA)


### 1.3 — Métriques de test recalculées en direct

In [4]:
y_prob_test = pipe_final.predict_proba(X_test)[:, 1]
y_pred_test = pipe_final.predict(X_test)  # seuil implicite 0.5, comme TP11
y_prob_tv   = pipe_final.predict_proba(X_tv)[:, 1]

pr_auc_train = average_precision_score(y_tv, y_prob_tv)
pr_auc_test  = average_precision_score(y_test, y_prob_test)
roc_auc_test = roc_auc_score(y_test, y_prob_test)
f1_test      = f1_score(y_test, y_pred_test, zero_division=0)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
precision_test = precision_score(y_test, y_pred_test, zero_division=0)
recall_test    = recall_score(y_test, y_pred_test, zero_division=0)

metrics = {
    "pr_auc_train": round(float(pr_auc_train), 4),
    "pr_auc_test":  round(float(pr_auc_test), 4),
    "roc_auc_test": round(float(roc_auc_test), 4),
    "f1_test":      round(float(f1_test), 4),
    "precision_test": round(float(precision_test), 4),
    "recall_test":    round(float(recall_test), 4),
    "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    "threshold": 0.5,
}
print(json.dumps(metrics, indent=2))
print(f"\nCohérence avec TP11.ipynb (PR-AUC test=0.8799, ROC-AUC=0.9949, F1=0.7586) : "
      f"{'OK' if abs(metrics['pr_auc_test'] - 0.8799) < 1e-3 else 'ECART'}")


{
  "pr_auc_train": 0.9997,
  "pr_auc_test": 0.8799,
  "roc_auc_test": 0.9949,
  "f1_test": 0.7586,
  "precision_test": 0.6681,
  "recall_test": 0.8776,
  "tp": 638,
  "tn": 18900,
  "fp": 317,
  "fn": 89,
  "threshold": 0.5
}

Cohérence avec TP11.ipynb (PR-AUC test=0.8799, ROC-AUC=0.9949, F1=0.7586) : OK


### 1.4 — Historique documenté (limites déjà connues)

Traçabilité indispensable pour une model card honnête : ce modèle est l'aboutissement
d'une lignée où une fuite de données a été détectée et corrigée.

In [5]:
LINEAGE = [
    {"model": "B5 (TP7)",    "note": "Baseline LogReg/RF/XGBoost, comparaison initiale"},
    {"model": "B7 (TP8)",    "note": "Optuna, PR-AUC val 0.8617 — FUITE via feature_row_id (ordre temporel)"},
    {"model": "B7c/B8 (TP8b)", "note": "Fuite corrigée — PR-AUC val 0.7504, split temporel"},
    {"model": "B8-ES (TP8c)", "note": "Early stopping — rejeté, pas d'amélioration"},
    {"model": "B9-GKF (TP9)", "note": "GroupKFold introduit — PR-AUC CV 0.7792"},
    {"model": "B11-GKF (TP11)", "note": "Optuna re-tuné sur objectif GroupKFold — RETENU"},
    {"model": "B12-GKF (TP12)", "note": "Feature stacking — rejeté, moins bon que B11"},
]
for row in LINEAGE:
    marker = "  <-- ce modèle" if "RETENU" in row["note"] else ""
    print(f"  {row['model']:<16} {row['note']}{marker}")


  B5 (TP7)         Baseline LogReg/RF/XGBoost, comparaison initiale
  B7 (TP8)         Optuna, PR-AUC val 0.8617 — FUITE via feature_row_id (ordre temporel)
  B7c/B8 (TP8b)    Fuite corrigée — PR-AUC val 0.7504, split temporel
  B8-ES (TP8c)     Early stopping — rejeté, pas d'amélioration
  B9-GKF (TP9)     GroupKFold introduit — PR-AUC CV 0.7792
  B11-GKF (TP11)   Optuna re-tuné sur objectif GroupKFold — RETENU  <-- ce modèle
  B12-GKF (TP12)   Feature stacking — rejeté, moins bon que B11


### 1.5 — Artefact persisté (TP9) — mise à jour depuis la première version de cette card

La première version de cette model card indiquait « aucun artefact persisté ». Depuis,
`DL/TP9.ipynb` a loggé B11-GKF dans MLflow (`ML/mlflow_tp7.db`, expérience
`TP7_maintenance_predictive`, run `XGBoost_GroupKFold_B11`) avec `mlflow.xgboost.log_model`.
On interroge le tracking store — pas un run_id recopié à la main — pour que cette
section reste correcte même si TP9 est rejoué et produit un nouveau run.


In [6]:
mlflow.set_tracking_uri(f"sqlite:///{Path('mlflow_tp7.db').resolve().as_posix()}")
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name("TP7_maintenance_predictive")

b11_runs = client.search_runs(
    experiment.experiment_id,
    filter_string="tags.mlflow.runName = 'XGBoost_GroupKFold_B11'",
    order_by=["start_time DESC"],
    max_results=1,
)

if b11_runs:
    mlflow_run = b11_runs[0]
    MLFLOW_RUN_ID = mlflow_run.info.run_id
    MLFLOW_MODEL_URI = f"runs:/{MLFLOW_RUN_ID}/xgboost_b11_gkf"
    print(f"Artefact MLflow trouvé : {MLFLOW_MODEL_URI}")
    print(f"PR-AUC test loggé      : {mlflow_run.data.metrics.get('pr_auc_test')}")
else:
    MLFLOW_RUN_ID = None
    MLFLOW_MODEL_URI = None
    print("Aucun run MLflow trouvé pour B11-GKF — exécuter DL/TP9.ipynb d'abord.")


Artefact MLflow trouvé : runs:/fa336bc0880b48bc849a4a6b1e6f412d/xgboost_b11_gkf
PR-AUC test loggé      : 0.8799


## §2 — Le template Hugging Face officiel

Même squelette que pour le modèle DL (`DL/TP8.ipynb`) — `ModelCard.from_template`.

In [7]:
template_head = Path(ModelCard.default_template_path).read_text(encoding="utf-8")
print(f"[{len(template_head.splitlines())} lignes, {template_head.count('{{')} champs {{{{ }}}} à remplir]")


[200 lignes, 45 champs {{ }} à remplir]


## §3 — Usage, biais, risques, limites

*Point de réflexion (Étape 3 du md)* : un lecteur non technique comprend-il **quand
faire confiance au modèle et quand se méfier** ? Le rappel modéré, la performance très
hétérogène selon les machines et l'historique de fuite corrigée sont documentés sans
euphémisme.

In [8]:
card_data = ModelCardData(
    model_name="indusense-xgb-maintenance-b11-gkf",
    license="other",
    library_name="xgboost",
    tags=["tabular-classification", "predictive-maintenance", "xgboost", "manufacturing", "time-series-features"],
    eval_results=[
        EvalResult(
            task_type="binary-classification",
            dataset_type="indusense-gold-machine-hourly",
            dataset_name="Gold machine hourly features — indusense_db",
            metric_type="pr_auc",
            metric_value=metrics["pr_auc_test"],
            metric_name="PR-AUC (average precision, test chronologique)",
        ),
    ],
)

template_kwargs = dict(
    model_id="indusense-xgb-maintenance-b11-gkf",
    model_summary=(
        "Classifieur XGBoost prédisant le risque de panne d'une machine industrielle "
        "dans les 24h à venir, à partir de features télémétrie (température, pression, "
        "tension, vitesse de rotation) et d'historique d'incidents agrégés sur des "
        "fenêtres glissantes 6h/12h/24h."
    ),
    model_description=(
        f"Entraîné sur {len(X_tv):,} observations horaires ({N_MACHINES} machines, "
        f"{N_FEATURES} features), évalué en validation croisée GroupKFold (une machine "
        "exclue par fold, pour ne jamais évaluer sur une machine vue à l'entraînement) "
        "et sur un jeu de test chronologiquement postérieur. Hyperparamètres optimisés "
        "par Optuna (TPE, 30 essais) sur l'objectif GroupKFold — voir `TP11.ipynb`."
    ),
    developers="Guillaume Saïdani",
    model_type="XGBoost (gradient boosting), classification binaire, pipeline scikit-learn (imputation médiane + XGBClassifier)",
    language="n/a (données tabulaires, pas de NLP)",
    license="Usage interne — données propriétaires (télémétrie machine, non publiques)",
    base_model="Aucun — entraîné from scratch",
    repo=(
        "ML/ (ce dépôt) — TP11.ipynb (recherche HP) ; artefact persisté depuis TP9 : "
        + (f"MLflow `{MLFLOW_MODEL_URI}` (store `ML/mlflow_tp7.db`)" if MLFLOW_MODEL_URI
           else "aucun — exécuter DL/TP9.ipynb pour en créer un")
    ),
    direct_use=(
        f"Scorer un enregistrement horaire machine (features télémétrie + historique "
        f"incidents agrégé) et obtenir une probabilité de panne à 24h. Seuil de "
        f"décision par défaut 0.5 (celui évalué ici) — à ajuster selon l'arbitrage "
        f"rappel/précision métier avant tout déploiement (cf. TP8 §8 pour une démarche "
        f"de calibration de seuil sur une version antérieure)."
    ),
    downstream_use=(
        "Alimentation d'un tableau de bord de maintenance prédictive classant les "
        "machines par risque décroissant, à l'usage d'un planificateur de maintenance "
        "humain — pas d'arrêt automatique de machine."
    ),
    out_of_scope_use=(
        "- Arrêt automatique ou décision de maintenance sans validation humaine.\n"
        "- Toute machine hors des 15 machines couvertes par l'entraînement, sans "
        "ré-entraînement (les performances varient déjà fortement d'une machine "
        "connue à l'autre, cf. limites).\n"
        "- Interprétation de `predict_proba` comme une probabilité calibrée — "
        "aucune calibration (Platt/isotonic) n'a été appliquée.\n"
        "- Usage réglementaire ou de certification sécurité — aucune validation de ce type."
    ),
    bias_risks_limitations=(
        f"- **Performance très hétérogène par machine** : PR-AUC en validation croisée "
        "GroupKFold va de 0.39 (MACH-07) à 1.00 (MACH-11, MACH-15) — écart-type "
        f"±{0.1891:.2f} autour d'une moyenne de {0.7839:.2f}. Un score agrégé unique "
        "masque des machines où le modèle est nettement moins fiable.\n"
        f"- **Sur-ajustement structurel** : PR-AUC train = {metrics['pr_auc_train']:.3f} "
        "contre ~0.78 en CV — piloté à 63% par une seule feature "
        "(`incident_max_severity_prev_24h`, diagnostic TP9/TP10). Persiste malgré une "
        "régularisation poussée (`reg_lambda`, `min_child_weight` élevés) ; qualifié de "
        "structurel, pas résolu par les hyperparamètres seuls.\n"
        "- **Historique de fuite de données** : une version antérieure (B7, TP8) "
        "incluait `feature_row_id`, un identifiant séquentiel corrélé à l'ordre "
        "temporel, qui gonflait le PR-AUC de +0.111. Corrigé depuis TP8b — retiré "
        "explicitement des `LEAKAGE_COLS` ci-dessus — mais signale la fragilité du "
        "pipeline de features aux fuites indirectes.\n"
        "- **Rappel modéré au seuil par défaut** : "
        f"{metrics['recall_test']:.1%} de rappel, {metrics['fn']} pannes non détectées "
        f"sur {metrics['fn']+metrics['tp']} au seuil 0.5 — un seuil plus bas augmenterait "
        "le rappel au prix de plus de fausses alertes (arbitrage non refait ici pour B11).\n"
        "- **Tentative de normalisation par machine infructueuse** : une normalisation "
        "z-score par machine a dégradé la généralisation en GroupKFold (TP10) — "
        "confirme que XGBoost est déjà insensible à l'échelle des features, ne pas "
        "réintroduire cette étape."
    ),
    bias_recommendations=(
        "Ne jamais utiliser en décision automatique. Suivre la performance par machine "
        "individuellement, pas seulement l'agrégat — une machine comme MACH-07 justifie "
        "une vigilance humaine renforcée plutôt qu'une confiance dans le score. Calibrer "
        "les probabilités (Platt/isotonic) avant tout usage nécessitant un score "
        "interprétable comme une probabilité réelle. Recalibrer le seuil de décision "
        "selon le coût métier faux négatif vs faux positif avant déploiement."
    ),
    get_started_code=(
        "```python\n"
        + (
            f"# Artefact persisté dans MLflow (voir §1.5) — rechargement direct\n"
            f"import mlflow.xgboost\n"
            f"mlflow.set_tracking_uri('sqlite:///mlflow_tp7.db')\n"
            f"model = mlflow.xgboost.load_model('{MLFLOW_MODEL_URI}')\n\n"
            f"from sklearn.impute import SimpleImputer\n"
            f"X_imputed = SimpleImputer(strategy='median').fit(X_train_val).transform(X_new)\n"
            f"proba = model.predict_proba(X_imputed)[:, 1]  # risque de panne à 24h\n"
            if MLFLOW_MODEL_URI else
            "# Aucun artefact persisté : ré-entraînement avec hyperparamètres figés (Optuna, TP11)\n"
            "from sklearn.pipeline import Pipeline\n"
            "from sklearn.impute import SimpleImputer\n"
            "from xgboost import XGBClassifier\n\n"
            "best_params = " + json.dumps(BEST_PARAMS) + "\n"
            "pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),\n"
            "                  ('model', XGBClassifier(**best_params))])\n"
            "pipe.fit(X_train_val, y_train_val)\n"
            "proba = pipe.predict_proba(X_new)[:, 1]  # risque de panne à 24h\n"
        )
        + "```"
    ),
)
print("card_data et template_kwargs (§3) prêts —", len(template_kwargs), "champs renseignés")


card_data et template_kwargs (§3) prêts — 15 champs renseignés


## §4 — Évaluation, impact environnemental, version, contact

In [9]:
template_kwargs.update(dict(
    training_data=(
        f"Table `gold_machine_hourly_feature` (PostgreSQL, base indusense_db) — "
        f"{len(X_tv):,} observations horaires (entraînement + validation), "
        f"{N_MACHINES} machines, {N_FEATURES} features (rolling 6h/12h/24h télémétrie "
        "+ historique incidents agrégé 24h/7j). Split chronologique (quantiles 0.70/0.85 "
        "sur `window_start`), pas de KFold aléatoire (fuite temporelle sinon)."
    ),
    preprocessing="Imputation des valeurs manquantes par médiane (`SimpleImputer`), aucune normalisation (XGBoost invariant à l'échelle — confirmé empiriquement, TP12).",
    training_regime=f"fp32, XGBoost gradient boosting, hyperparamètres Optuna (TPE, 30 essais, objectif GroupKFold(5)), scale_pos_weight={SPW_GLOBAL} (déséquilibre de classes)",
    speeds_sizes_times=f"{carbon_data.duration:.1f}s pour un ré-entraînement complet sur {len(X_tv):,} lignes (poste de travail local, CPU)",
    testing_data=f"Même table, partition test chronologiquement postérieure — {len(X_test):,} lignes, {int(y_test.sum())} pannes positives.",
    testing_factors="Évaluation agrégée toutes machines confondues pour la métrique principale ; performance par machine disponible via validation croisée GroupKFold (voir §3, hétérogénéité 0.39-1.00).",
    testing_metrics="PR-AUC (average precision — préférée à l'accuracy vu le déséquilibre de classe), ROC-AUC, F1, matrice de confusion au seuil 0.5.",
    results=(
        f"PR-AUC train={metrics['pr_auc_train']:.4f} · PR-AUC test={metrics['pr_auc_test']:.4f} · "
        f"ROC-AUC test={metrics['roc_auc_test']:.4f} · F1 test={metrics['f1_test']:.4f} · "
        f"TP={metrics['tp']} TN={metrics['tn']} FP={metrics['fp']} FN={metrics['fn']} · "
        f"Précision={metrics['precision_test']:.1%} · Rappel={metrics['recall_test']:.1%}"
    ),
    results_summary=(
        f"PR-AUC test {metrics['pr_auc_test']:.2f} sur un problème fortement déséquilibré "
        f"(scale_pos_weight={SPW_GLOBAL:.0f}) — signal réel mais hétérogène selon les "
        "machines (voir limites). Écart train/CV important : le modèle généralise moins "
        "bien qu'il ne le suggère sur ses propres données d'entraînement."
    ),
    model_examination="Non réalisé dans ce notebook — SHAP (TreeExplainer) recommandé en prochaine étape pour identifier les features dominantes par machine (cf. b7_optimisation_explicabilite.md, hors périmètre ici).",
    hardware_type="Intel Core i7-12700H (CPU) — entraînement XGBoost, pas de GPU requis",
    hours_used=f"{carbon_data.duration/3600:.4f} h (ré-entraînement complet mesuré ci-dessus)",
    cloud_provider="Aucun — poste de travail local",
    cloud_region=f"{carbon_data.country_name} ({carbon_data.country_iso_code})",
    co2_emitted=f"{emissions_kg*1000:.4f} gCO2eq ({carbon_data.energy_consumed*1000:.3f} Wh) — mesure CodeCarbon de ce ré-entraînement (première mesure du genre côté ML, introduite dans la version précédente de cette card)",
    model_specs=f"XGBoost : n_estimators=287, max_depth=9, learning_rate={BEST_PARAMS['learning_rate']:.4f}, objectif binaire pondéré (scale_pos_weight={SPW_GLOBAL})",
    compute_infrastructure=(
        "Poste de travail local, connexion PostgreSQL directe. "
        + (f"Artefact tracké et versionné dans MLflow depuis TP9 ({MLFLOW_MODEL_URI})."
           if MLFLOW_MODEL_URI else
           "Aucune pipeline CLI industrialisée pour ce modèle (contrairement au DL, cf. DL/TP7.ipynb).")
    ),
    hardware_requirements="CPU suffisant — pas de dépendance GPU",
    software="xgboost 3.3.0, scikit-learn 1.9.0, optuna 4.9.0, mlflow 3.14.0, Python 3.13 (ML/.venv)",
    model_card_authors="Guillaume Saïdani",
    model_card_contact="guillaume.saidani@ext.aelion.fr",
))

card = ModelCard.from_template(card_data, **template_kwargs)
card.save(ARTIFACTS_DIR / "model_card.md")
print(f"Model card sauvegardée : {ARTIFACTS_DIR / 'model_card.md'}")
print(f"Longueur : {len(str(card))} caractères")


Model card sauvegardée : artifacts\model_card.md
Longueur : 11361 caractères


In [10]:
card.validate()
print("Validation OK — front-matter YAML conforme au schéma Hugging Face")
print()
print(str(card)[:1500])
print("...")


Validation OK — front-matter YAML conforme au schéma Hugging Face

---
library_name: xgboost
license: other
tags:
- tabular-classification
- predictive-maintenance
- xgboost
- manufacturing
- time-series-features
model-index:
- name: indusense-xgb-maintenance-b11-gkf
  results:
  - task:
      type: binary-classification
    dataset:
      name: Gold machine hourly features — indusense_db
      type: indusense-gold-machine-hourly
    metrics:
    - type: pr_auc
      value: 0.8799
      name: PR-AUC (average precision, test chronologique)
---

# Model Card for indusense-xgb-maintenance-b11-gkf

<!-- Provide a quick summary of what the model is/does. -->

Classifieur XGBoost prédisant le risque de panne d'une machine industrielle dans les 24h à venir, à partir de features télémétrie (température, pression, tension, vitesse de rotation) et d'historique d'incidents agrégés sur des fenêtres glissantes 6h/12h/24h.

## Model Details

### Model Description

<!-- Provide a longer summary of wh

## Synthèse

**Mise à jour (post-TP9)** : B11-GKF a désormais un **artefact persisté** dans MLflow
(`ML/mlflow_tp7.db`, run `XGBoost_GroupKFold_B11`), loggé par `DL/TP9.ipynb`. Cette
card interroge le tracking store à chaque exécution (§1.5) plutôt que de coder en dur
un `run_id` — elle reste correcte même si TP9 est rejoué.

**Honnêteté délibérée** : la card documente l'historique de fuite de données corrigée
(B7 → B7c) et l'hétérogénéité de performance par machine (0.39 à 1.00 en PR-AUC) plutôt
que de ne présenter que le score agrégé flatteur (PR-AUC test 0.88).

**Premier chiffre carbone pour le modèle ML** : aucune mesure CodeCarbon n'existait
avant cette card côté maintenance — comblé en §1.2.
